# GENCODE and raw enrichment with provenance

Executed examples use tiny synthetic GTF/Parquet and HTTP-boundary JSON
fixtures. No live datasets, language models or structure predictors run.
`gencode.get_genes` retains **all biotypes**; `fetch_mane_select` selects the
protein-coding/MANE subset. EuropePMC and structure clients cache successful
responses, including valid absence, but retry failed inputs/providers.


In [1]:
import gzip
import json
import tempfile
from pathlib import Path
from unittest.mock import patch

import polars as pl
import requests

work = Path(tempfile.mkdtemp(prefix="biodb-tutorial-"))

# A tiny HTTP boundary fixture: bioDB still performs its real parsing,
# validation, durable writes, and cache/receipt checks.


class Response:
    def __init__(self, data=None, body=None, status=200):
        self.data = data
        self.content = body if body is not None else json.dumps(data).encode()
        self.text = self.content.decode(errors="replace")
        self.status_code = status
        self.headers = {"Content-Length": str(len(self.content)), "ETag": '"fixture-v1"'}

    def json(self):
        return self.data

    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(f"synthetic HTTP {self.status_code}", response=self)

    def iter_content(self, chunk_size=65536):
        for start in range(0, len(self.content), chunk_size):
            yield self.content[start : start + chunk_size]

    def __enter__(self):
        return self

    def __exit__(self, *args):
        pass


class Session:
    def __init__(self, replies):
        self.replies, self.calls = iter(replies), []

    def get(self, url, **kwargs):
        self.calls.append(url)
        reply = next(self.replies)
        if isinstance(reply, Exception):
            raise reply
        return reply

In [2]:
from biodb import gencode

release = "47"
lines = [
    'chr1\tfixture\tgene\t1000\t2000\t.\t+\t.\tgene_id "ENSG_X.5"; gene_name "GENEX"; gene_type "protein_coding";',
    'chr1\tfixture\ttranscript\t1000\t2000\t.\t+\t.\tgene_id "ENSG_X.5"; transcript_id "ENST_X.3"; transcript_type "protein_coding"; tag "MANE_Select";',
    'chr1\tfixture\tCDS\t1001\t1100\t.\t+\t0\tgene_id "ENSG_X.5"; transcript_id "ENST_X.3"; tag "MANE_Select";',
    'chr1\tfixture\tCDS\t1201\t1250\t.\t+\t0\tgene_id "ENSG_X.5"; transcript_id "ENST_X.3"; tag "MANE_Select";',
    'chr1\tfixture\tgene\t11\t20\t.\t-\t.\tgene_id "ENSG_R.2"; gene_name "RNA1"; gene_type "lncRNA";',
]
# Populate the documented pooch cache filename with a real compressed GTF.
(work / f"gencode.v{release}.annotation.gtf.gz").write_bytes(
    gzip.compress(("\n".join(lines) + "\n").encode())
)
genes = gencode.get_genes(release, cache_dir=work)
mane = gencode.fetch_mane_select(release, cache_dir=work)
assert set(genes["biotype"]) == {"protein_coding", "lncRNA"}
assert genes.filter(pl.col("gene_id") == "ENSG_R")["start"][0] == 10
assert mane["aa_length"][0] == 49  # (150 CDS bases - 3) // 3, current API convention
print(genes)
print(mane.select("gene_id", "mane_select_tx", "aa_length"))

shape: (2, 7)
┌─────────┬─────────────┬────────────────┬───────┬───────┬──────┬────────┐
│ gene_id ┆ gene_symbol ┆ biotype        ┆ chrom ┆ start ┆ end  ┆ strand │
│ ---     ┆ ---         ┆ ---            ┆ ---   ┆ ---   ┆ ---  ┆ ---    │
│ str     ┆ str         ┆ str            ┆ str   ┆ i64   ┆ i64  ┆ str    │
╞═════════╪═════════════╪════════════════╪═══════╪═══════╪══════╪════════╡
│ ENSG_X  ┆ GENEX       ┆ protein_coding ┆ chr1  ┆ 999   ┆ 2000 ┆ +      │
│ ENSG_R  ┆ RNA1        ┆ lncRNA         ┆ chr1  ┆ 10    ┆ 20   ┆ -      │
└─────────┴─────────────┴────────────────┴───────┴───────┴──────┴────────┘
shape: (1, 3)
┌─────────┬────────────────┬───────────┐
│ gene_id ┆ mane_select_tx ┆ aa_length │
│ ---     ┆ ---            ┆ ---       │
│ str     ┆ str            ┆ i64       │
╞═════════╪════════════════╪═══════════╡
│ ENSG_X  ┆ ENST_X.3       ┆ 49        │
└─────────┴────────────────┴───────────┘


GENCODE gene coordinates are **zero-based, half-open**; GTF input is one-based
inclusive. Gene IDs lose their version suffix; the MANE transcript ID retains
its version. The synthetic cache is deliberately isolated and must not be used
as a real annotation release.


In [3]:
from biodb import europepmc, protein_structure

first = Session(
    [
        Response(
            {"resultList": {"result": [{"id": "1", "title": "synthetic title", "extraField": 7}]}}
        ),
        ConnectionError("synthetic network interruption"),
    ]
)
try:
    europepmc.fetch_metadata(
        ["1", "2"], cache_dir=work / "literature", batch_size=1, max_retries=1, session=first
    )
except ConnectionError:
    print("Interrupted after PMID 1 committed.")
second = Session([Response({"resultList": {"result": []}})])
metadata = europepmc.fetch_metadata(
    ["1", "2"], cache_dir=work / "literature", batch_size=1, session=second
)
assert metadata["1"]["extraField"] == 7 and len(second.calls) == 1
assert europepmc.fetch_metadata(["2"], cache_dir=work / "literature", session=Session([])) == {}
print(metadata, "; resume requested only the failed PMID:", len(second.calls))

first = Session(
    [
        Response([{"entryId": "AF-P1", "providerExtra": {"quality": "synthetic"}}]),
        ConnectionError("synthetic RCSB failure"),
    ]
)
try:
    protein_structure.fetch_structure_for_accession(
        "P1", cache_dir=work / "structures", session=first
    )
except ConnectionError:
    print("AlphaFold committed; RCSB still incomplete.")
second = Session([Response(status=404)])
structure = protein_structure.fetch_structure_for_accession(
    "P1", cache_dir=work / "structures", session=second
)
assert structure["rcsb"] == [] and len(second.calls) == 1
assert structure["alphafold"][0]["providerExtra"]["quality"] == "synthetic"
print(structure)

Interrupted after PMID 1 committed.
{'1': {'id': '1', 'title': 'synthetic title', 'extraField': 7}} ; resume requested only the failed PMID: 1
AlphaFold committed; RCSB still incomplete.
{'uniprot_accession': 'P1', 'alphafold': [{'entryId': 'AF-P1', 'providerExtra': {'quality': 'synthetic'}}], 'rcsb': []}


In [4]:
import io
from contextlib import redirect_stderr

from biodb import opentargets as ot
from biodb.opentargets.literature import fetch_literature_matches

stream = io.BytesIO()
pl.DataFrame(
    [
        {
            "type": "CD",
            "keywordId": "D1",
            "pmid": "1",
            "text": "source sentence",
            "section": "title",
            "unmodeled": "preserved",
        },
        {
            "type": "GP",
            "keywordId": "D1",
            "pmid": "1",
            "text": "not selected",
            "section": "title",
            "unmodeled": "also raw",
        },
    ]
).write_parquet(stream)
session = Session(
    [Response(body=b'<a href="part.parquet">part</a>'), Response(body=stream.getvalue())]
)

with redirect_stderr(io.StringIO()):
    matches = fetch_literature_matches(
        ot.DEFAULT_VERSION, {("D1", "1")}, cache_dir=work / "ot", session=session
    )
assert len(matches) == 1 and matches[0]["unmodeled"] == "preserved"
print(matches)
# Cache source identity retains the intermediate path, not an invented output dataset.
manifest = next((work / "ot").rglob(".biodb-manifest.json"))
print("source:", json.loads(manifest.read_text())["identity"]["source_url"])

[{'type': 'CD', 'keywordId': 'D1', 'pmid': '1', 'text': 'source sentence', 'section': 'title', 'unmodeled': 'preserved'}]
source: https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/25.12/intermediate/literature_match


In [5]:
import base64

from biodb.opentargets.constants import fetch_constants

variant_ts = 'const VARIANT_CONSEQUENCES: VariantConsequence[] = [\n {id: "SO:0001583", label: "missense_variant"}\n];\nconst VARIANT_EFFECT_METHODS: Record<string, VariantEffectMethod> = {\n AlphaMissense: { methodName: "AlphaMissense", prettyName: "AlphaMissense" }\n};\n'
profile_ts = 'const populationLabels = { nfe_adj: "Non-Finnish European" };'
index_ts = 'export const clinvarStarMap: Record<string, number> = {"reviewed": 3};\nexport const credsetConfidenceMap: Record<string, number> = {"high": 1};'


def source_response(text, sha):
    return Response({"content": base64.b64encode(text.encode()).decode(), "sha": sha})


session = Session(
    [
        source_response(variant_ts, "fixture-variant-sha"),
        source_response(profile_ts, "fixture-profile-sha"),
        source_response(index_ts, "fixture-index-sha"),
    ]
)
constants = fetch_constants(ref="fixture-commit", cache_dir=work / "constants", session=session)
assert constants["variant_consequences"] == {"SO_0001583": "missense_variant"}
assert (
    fetch_constants(ref="fixture-commit", cache_dir=work / "constants", session=Session([]))
    == constants
)
print(constants)

{'variant_consequences': {'SO_0001583': 'missense_variant'}, 'variant_effect_methods': {'AlphaMissense': 'AlphaMissense'}, 'population_labels': {'nfe_adj': 'Non-Finnish European'}, 'clinvar_star_map': {'reviewed': 3}, 'credset_confidence_map': {'high': 1}, 'variant_ts_sha': 'fixture-variant-sha', 'profile_header_sha': 'fixture-profile-sha', 'index_ts_sha': 'fixture-index-sha'}


In [6]:
from biodb import ols_text

entity = {
    "curie": "SNOMED:1",
    "iri": "http://snomed.info/id/1",
    "label": ["Preferred (finding)", "Preferred"],
    ols_text.SKOS + "prefLabel": "Preferred",
    ols_text.SKOS + "altLabel": ["Alternative", {"value": "Second alias"}],
    "http://purl.obolibrary.org/obo/IAO_0000115": ["First definition", "Second definition"],
    "linkedEntities": {"SNOMED:2": {"label": "Neighbor label must not become text"}},
}
record = ols_text.text_record(entity)
assert record["synonyms"] == ["Preferred (finding)", "Alternative", "Second alias"]
assert record["description"] == ["First definition", "Second definition"]
metadata = {"ontologyId": "snomed", "loaded": "synthetic-snapshot", "numberOfClasses": 1}


def ols_response(url, **kwargs):
    if url.endswith("/classes"):
        return Response({"totalPages": 1, "totalElements": 1, "elements": [entity]})
    return Response(metadata)


with patch("requests.get", ols_response):
    audit = ols_text.download_text_cache(
        "snomed",
        work / "concepts.parquet",
        page_cache=work / "ols_pages",
        workers=1,
        expected_ids=["SNOMED:1"],
    )
print(pl.read_parquet(work / "concepts.parquet"))
print("audited source snapshot:", audit["source_metadata"]["loaded"])

shape: (1, 6)
┌──────────┬───────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┐
│ obo_id   ┆ label     ┆ iri                   ┆ description  ┆ synonyms             ┆ is_obsolete │
│ ---      ┆ ---       ┆ ---                   ┆ ---          ┆ ---                  ┆ ---         │
│ str      ┆ str       ┆ str                   ┆ list[str]    ┆ list[str]            ┆ bool        │
╞══════════╪═══════════╪═══════════════════════╪══════════════╪══════════════════════╪═════════════╡
│ SNOMED:1 ┆ Preferred ┆ http://snomed.info/id ┆ ["First      ┆ ["Preferred          ┆ false       │
│          ┆           ┆ /1                    ┆ definition", ┆ (finding)", "Alter…  ┆             │
│          ┆           ┆                       ┆ "Second d…   ┆                      ┆             │
└──────────┴───────────┴───────────────────────┴──────────────┴──────────────────────┴─────────────┘
audited source snapshot: synthetic-snapshot


## Live production recipes — not executed

Raw enrichment retains provider JSON and upstream row fields. Rendering,
document ranking and biological interpretation belong in downstream code.
For reproducibility pin the OT data version, GitHub commit and GENCODE release;
retain returned source SHAs and cache provenance. Valid provider 404/empty
results differ from failures (500/timeouts); failures never prove absence.

```python
from biodb.opentargets.constants import fetch_constants, fetch_sequence_ontology_labels
constants = fetch_constants(ref="<ot-ui-commit>", cache_dir="cache/constants")
so_labels = fetch_sequence_ontology_labels(version=ot.DEFAULT_VERSION, cache_dir="cache/ot")
metadata = europepmc.fetch_metadata(["30700171"], cache_dir="cache/europepmc")
structure = protein_structure.fetch_structure_for_accession("P38398", cache_dir="cache/structures")
genes = gencode.get_genes("47", cache_dir="cache/gencode")
# Full OLS text acquisition may be large. It audits count, duplicate IDs,
# expected concept membership, and unchanged source metadata before publishing.
from biodb import snomed
snomed.bulk_text_concepts("snomed.parquet", page_cache="snomed-pages")
```

`ontology_owl` is a deprecated compatibility shim: use `biodb.ontology`.
OLS text fields preserve literal definitions and synonyms; graph neighbors do
not supply invented prose. OLS page-cache snapshot/size changes require a new
page directory. See [ontology](04_ontology.ipynb), [PubMed](11_pubmed.ipynb), and
[SNOMED](12_snomed.ipynb) for the complementary APIs.
